# Closure and transformation tests

In [21]:
import numpy as np
from openfermion import FermionOperator, jordan_wigner, get_sparse_operator
from src.utils.transform_utils import *
from src.utils.op_utils import *
import time


G = (-1.j)*Tia(0, 1)
H = FermionOperator('3^ 2 1^ 1', 1.0) + FermionOperator('1^ 1 2^ 3', 1.0) + FermionOperator('6^ 2 1^ 1', 1.0) + FermionOperator('1^ 1 2^ 6', 1.0) 
n_qubits = max(count_qubits(G), count_qubits(H))

#BRUTE FORCE S

start_time = time.perf_counter()
e, S = get_S(G, H, True)
end_time = time.perf_counter()

elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.6f} seconds")

G: 1j [0^ 2] +
1j [1^ 3] +
-1j [2^ 0] +
-1j [3^ 1]
H: 1.0 [1^ 1 2^ 3] +
1.0 [1^ 1 2^ 6] +
1.0 [3^ 2 1^ 1] +
1.0 [6^ 2 1^ 1]
Block norms:
 [[0.         1.         1.41421356 1.         0.        ]
 [1.         2.         2.44948974 2.         1.        ]
 [1.41421356 2.44948974 2.82842712 2.44948974 1.41421356]
 [1.         2.         2.44948974 2.         1.        ]
 [0.         1.         1.41421356 1.         0.        ]]
Eigen values:
 [-2. -1. -0.  1.  2.]
Eigen value differences:
 [[ 0.+0.j -1.+0.j -2.+0.j -3.+0.j -4.+0.j]
 [ 1.+0.j  0.+0.j -1.+0.j -2.+0.j -3.+0.j]
 [ 2.+0.j  1.+0.j  0.+0.j -1.+0.j -2.+0.j]
 [ 3.+0.j  2.+0.j  1.+0.j  0.+0.j -1.+0.j]
 [ 4.+0.j  3.+0.j  2.+0.j  1.+0.j  0.+0.j]]
Found 7 unique eigenvalue differences over non-vanishing blocks:
(maximum possible differences: 9) 
[0.0, 1.0, 2.0, 3.0, -2.0, -3.0, -1.0]
Execution time: 0.154392 seconds


In [24]:
### Find S with nested commutator sums and powers of G
from src.utils.transform_utils import find_S_blocks, find_S_commutators

start_time = time.perf_counter()
S = find_S_blocks(G, H)
end_time = time.perf_counter()
print("\nS obtained from evaluating projected blocks: ", S)

elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.6f} seconds")

start_time = time.perf_counter()
S = find_S_commutators(G, H)
end_time = time.perf_counter()
print("\n\nS obtained from evaluating commutators: ", S)

elapsed_time = end_time - start_time
print(f"Execution time: {elapsed_time:.6f} seconds")


S obtained from evaluating projected blocks:  [0.0, 1.0, 2.0, 3.0, -2.0, -3.0, -1.0]
Execution time: 0.057996 seconds


S obtained from evaluating commutators:  [0.0, 1.0, 2.0, 3.0, -2.0, -3.0, -1.0]
Execution time: 0.041870 seconds


### check commutator closure with S

In [25]:
G_pauli = jordan_wigner(G)
H_pauli = jordan_wigner(H)
commutators = get_commutators(G_pauli, H_pauli, len(S))
coeff = get_poly_coeff(S)

fh = np.sum([c*t for c, t in zip(coeff, commutators)])
print(f'f(H) = {fh}')

f(H) = 0


### test transformation

In [26]:
import scipy.sparse as sp

# perform transform
theta = np.pi/3

#get exact coeffs
e_vec = np.exp(1.j * theta * np.array(S))
W = get_vander(S)
a = np.linalg.inv(W) @ e_vec
comm_sum = sum([ai*comm for ai, comm in zip(a, commutators)])

# full operator for verification
U = sp.linalg.expm(1.j * get_sparse_operator(G, n_qubits) * theta)
UHUd = U @ get_sparse_operator(H, n_qubits) @ U.getH()

tol =1e-5
assert np.sum(np.abs(get_sparse_operator(comm_sum, n_qubits) - UHUd)) < tol

## examples in Table 1

In [27]:
# case [-2, 0, 2] (a)
G = (-1.j)*(FermionOperator('2^ 1^ 2 0', 1.0) - FermionOperator('0^ 2^ 1 2', 1.0))
H = FermionOperator('3^ 1^ 4 0', 1.0) + FermionOperator('0^ 4^ 1 3', 1.0)
e, S = get_S(G, H, True)

G: 1j [0^ 2^ 1 2] +
-1j [2^ 1^ 2 0]
H: 1.0 [0^ 4^ 1 3] +
1.0 [3^ 1^ 4 0]
Block norms:
 [[0.70710678 0.         0.70710678]
 [0.         1.41421356 0.        ]
 [0.70710678 0.         0.70710678]]
Eigen values:
 [-1.  0.  1.]
Eigen value differences:
 [[ 0.+0.j -1.+0.j -2.+0.j]
 [ 1.+0.j  0.+0.j -1.+0.j]
 [ 2.+0.j  1.+0.j  0.+0.j]]
Found 3 unique eigenvalue differences over non-vanishing blocks:
(maximum possible differences: 5) 
[0.0, 2.0, -2.0]


In [28]:
# case [-2, 0, 2] (b)
G = (-1.j)*(FermionOperator('3^ 2^ 1 0', 1.0) - FermionOperator('0^ 1^ 2 3', 1.0))
H = FermionOperator('1^ 1', 1.0)
e, S = get_S(G, H, True)

G: 1j [0^ 1^ 2 3] +
-1j [3^ 2^ 1 0]
H: 1.0 [1^ 1]
Block norms:
 [[0.5        0.         0.5       ]
 [0.         2.64575131 0.        ]
 [0.5        0.         0.5       ]]
Eigen values:
 [-1.  0.  1.]
Eigen value differences:
 [[ 0.+0.j -1.+0.j -2.+0.j]
 [ 1.+0.j  0.+0.j -1.+0.j]
 [ 2.+0.j  1.+0.j  0.+0.j]]
Found 3 unique eigenvalue differences over non-vanishing blocks:
(maximum possible differences: 5) 
[0.0, 2.0, -2.0]


In [29]:
# case [-2, 2]
G = (-1.j)*(FermionOperator('3^ 2^ 1 0', 1.0) - FermionOperator('0^ 1^ 2 3', 1.0))
H = FermionOperator('3^ 2^ 1 0', 1.0) + FermionOperator('0^ 1^ 2 3', 1.0)
e, S = get_S(G, H, True)

G: 1j [0^ 1^ 2 3] +
-1j [3^ 2^ 1 0]
H: 1.0 [0^ 1^ 2 3] +
1.0 [3^ 2^ 1 0]
Block norms:
 [[0. 0. 1.]
 [0. 0. 0.]
 [1. 0. 0.]]
Eigen values:
 [-1.  0.  1.]
Eigen value differences:
 [[ 0.+0.j -1.+0.j -2.+0.j]
 [ 1.+0.j  0.+0.j -1.+0.j]
 [ 2.+0.j  1.+0.j  0.+0.j]]
Found 2 unique eigenvalue differences over non-vanishing blocks:
(maximum possible differences: 5) 
[2.0, -2.0]


In [30]:
# case [-1, 1]
G = (-1.j)*(FermionOperator('3^ 2^ 1 0', 1.0) - FermionOperator('0^ 1^ 2 3', 1.0))
H = FermionOperator('4^ 2^ 1 0', 1.0) + FermionOperator('0^ 1^ 2 4', 1.0)
e, S = get_S(G, H, True)

G: 1j [0^ 1^ 2 3] +
-1j [3^ 2^ 1 0]
H: 1.0 [0^ 1^ 2 4] +
1.0 [4^ 2^ 1 0]
Block norms:
 [[0. 1. 0.]
 [1. 0. 1.]
 [0. 1. 0.]]
Eigen values:
 [-1.  0.  1.]
Eigen value differences:
 [[ 0.+0.j -1.+0.j -2.+0.j]
 [ 1.+0.j  0.+0.j -1.+0.j]
 [ 2.+0.j  1.+0.j  0.+0.j]]
Found 2 unique eigenvalue differences over non-vanishing blocks:
(maximum possible differences: 5) 
[1.0, -1.0]


In [31]:
# case [-1, 0, 1]
G = (-1.j)*(FermionOperator('3^ 2^ 1 0', 1.0) - FermionOperator('0^ 1^ 2 3', 1.0))
H = FermionOperator('3^ 2^ 2 0', 1.0) + FermionOperator('0^ 2^ 2 3', 1.0)
e, S = get_S(G, H, True)

G: 1j [0^ 1^ 2 3] +
-1j [3^ 2^ 1 0]
H: 1.0 [0^ 2^ 2 3] +
1.0 [3^ 2^ 2 0]
Block norms:
 [[0.         0.70710678 0.        ]
 [0.70710678 1.41421356 0.70710678]
 [0.         0.70710678 0.        ]]
Eigen values:
 [-1.  0.  1.]
Eigen value differences:
 [[ 0.+0.j -1.+0.j -2.+0.j]
 [ 1.+0.j  0.+0.j -1.+0.j]
 [ 2.+0.j  1.+0.j  0.+0.j]]
Found 3 unique eigenvalue differences over non-vanishing blocks:
(maximum possible differences: 5) 
[0.0, 1.0, -1.0]
